In [16]:
# file: scripts/multiclass_model_zoo_kfold.py
# -*- coding: utf-8 -*-
"""
多模型多分類 + 二元(head) 對比（50k cells / 9 類）
- StratifiedKFold / GroupKFold
- 多類與二元同時訓練：多類負責 argmax；二元提供 p_pos
- 固定門檻 0.55：用二元 p_pos 覆寫為 POS_LABEL（mask）
- 模型：LogReg(softmax), Ridge(校準), LinearSVC(校準)
"""
import os
import numpy as np
import pandas as pd
from typing import Tuple, Dict, List, Optional
from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    adjusted_rand_score, v_measure_score
)
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline

# ---------------- Config ----------------
IN_DIR = "../outputs_features_csv"
FILES = {"Combined (z-score)": "features_combined_z-score.csv"}

LABEL_COL = "cell_type"
POS_LABEL = "T"                 # 二元 head 的正類
THRESH_FIXED = 0.55             # 固定覆寫門檻
OPTIONAL_GROUP_COLS = ["Patient", "Sample", "set"]

N_SPLITS  = 5
SEED      = 42
USE_SAMPLE_WEIGHTS = True
WEIGHT_SMOOTH_ALPHA = 0.0
WEIGHT_MAX = 5.0

MODEL_CHOICES = [
    "logreg_softmax",
    "ridge_ovr",
    "linearsvc_calibrated",
]

ADD_STANDARD_SCALER = False  # 目前特徵已 z-score

# ---------------- Utils ----------------
def load_full(path: str) -> pd.DataFrame:
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    df = pd.read_csv(path, index_col=0)
    if df.empty:
        raise ValueError(f"Empty: {path}")
    return df

def split_masks(df: pd.DataFrame, label_col: str) -> Tuple[pd.Series, pd.Series]:
    if label_col not in df.columns:
        return pd.Series(False, index=df.index), pd.Series(True, index=df.index)
    lbl = df[label_col]
    is_lab = (lbl.notna()
              & lbl.astype(str).str.strip().ne("")
              & lbl.astype(str).str.lower().ne("nan"))
    return is_lab, ~is_lab

def numeric_feature_matrix(df: pd.DataFrame) -> pd.DataFrame:
    Xdf = df.select_dtypes(include=[np.number]).copy()
    nunique = Xdf.nunique(dropna=True)
    keep = (nunique > 1)
    Xdf = Xdf.loc[:, keep]
    if Xdf.shape[1] == 0:
        raise ValueError("No usable numeric features (all constant/NaN).")
    return Xdf.fillna(0.0)

def groups_from(df: pd.DataFrame, cols: List[str]) -> Optional[np.ndarray]:
    for c in cols:
        if c in df.columns:
            return df[c].astype(str).values
    return None

def make_sample_weights(y_like: np.ndarray,
                        smooth_alpha: float = WEIGHT_SMOOTH_ALPHA,
                        max_w: Optional[float] = WEIGHT_MAX) -> np.ndarray:
    vc = pd.Series(y_like).value_counts()
    K = float(vc.shape[0]); N = float(len(y_like))
    denom = vc + smooth_alpha
    w_c = (N / (K * denom)).to_dict()
    w = np.array([w_c[y] for y in y_like], dtype=float)
    if max_w is not None:
        w = np.minimum(w, float(max_w))
    return w

def metrics_all(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    return dict(
        ACC=accuracy_score(y_true, y_pred),
        Balanced_ACC=balanced_accuracy_score(y_true, y_pred),
        F1_macro=f1_score(y_true, y_pred, average="macro"),
        ARI=adjusted_rand_score(y_true, y_pred),
        V=v_measure_score(y_true, y_pred),
    )

# ---------------- Model builders ----------------
def _scaler():
    return [("scaler", StandardScaler(with_mean=False))] if ADD_STANDARD_SCALER else []

def _make_calibrator(estimator) -> CalibratedClassifierCV:
    """sklearn 版本兼容：優先用 estimator=，失敗則改 base_estimator="""
    try:
        return CalibratedClassifierCV(estimator=estimator, method="sigmoid", cv=3)
    except TypeError:
        return CalibratedClassifierCV(base_estimator=estimator, method="sigmoid", cv=3)

def build_model(name: str, *, for_binary: bool = False) -> Pipeline:
    """
    回傳『可輸出機率』的分類器（多類/二元皆可）。
    """
    if name == "logreg_softmax":
        if for_binary:
            clf = LogisticRegression(
                penalty="l2", C=1.0, solver="lbfgs",
                max_iter=200, n_jobs=-1, random_state=SEED
            )
        else:
            clf = LogisticRegression(
                penalty="l2", C=1.0, solver="lbfgs",
                multi_class="multinomial", max_iter=200, n_jobs=-1, random_state=SEED
            )
        return Pipeline(_scaler() + [("clf", clf)])

    if name == "ridge_ovr":
        base = RidgeClassifier(alpha=1.0, random_state=None)
        clf = _make_calibrator(base)  # 版本安全
        return Pipeline(_scaler() + [("clf", clf)])

    if name == "linearsvc_calibrated":
        base = LinearSVC(C=1.0, class_weight=None, max_iter=5000, random_state=SEED)
        clf = _make_calibrator(base)  # 版本安全
        return Pipeline(_scaler() + [("clf", clf)])

    raise ValueError(f"Unknown model name: {name}")

def _fit_with_optional_weight(clf: Pipeline, X, y, sample_weight: Optional[np.ndarray]):
    """對不同 sklearn 版本安全傳遞 sample_weight；不支援則忽略。"""
    if sample_weight is None:
        clf.fit(X, y)
        return
    try:
        clf.fit(X, y, **{"clf__sample_weight": sample_weight})
    except TypeError:
        clf.fit(X, y)

def _predict_proba(clf: Pipeline, X: np.ndarray) -> np.ndarray:
    # CalibratedClassifierCV & LogisticRegression 都有 predict_proba
    return clf.predict_proba(X)

# ---------------- Runner ----------------
def run_kfold_and_predict(df: pd.DataFrame, fname: str) -> Dict[str, pd.DataFrame]:
    is_lab, is_unlab = split_masks(df, LABEL_COL)
    if not is_lab.any():
        raise ValueError(f"{fname}: no labeled rows for training.")

    X_all = numeric_feature_matrix(df).to_numpy(dtype=float)
    y_str_all = df[LABEL_COL].astype(str).values
    idx_all = df.index.to_numpy()

    # labeled / unlabeled
    X_lab = X_all[is_lab.values]
    y_str = y_str_all[is_lab.values]
    idx_lab = idx_all[is_lab.values]

    X_unlab = X_all[is_unlab.values]
    idx_unlab = idx_all[is_unlab.values]

    # encoders & constants
    le = LabelEncoder().fit(y_str)
    classes = le.classes_
    if POS_LABEL not in classes:
        raise ValueError(f"POS_LABEL '{POS_LABEL}' 不在類別中：{classes.tolist()}")
    y_multi = le.transform(y_str)
    y_bin = (y_str == POS_LABEL).astype(int)
    K = len(classes)
    idx_T = int(np.where(classes == POS_LABEL)[0][0])

    # groups（labeled only）
    groups = groups_from(df.loc[idx_lab], OPTIONAL_GROUP_COLS)

    # split
    if groups is not None:
        splitter = GroupKFold(n_splits=N_SPLITS)
        fold_iter = list(splitter.split(X_lab, y_multi, groups))
    else:
        splitter = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
        fold_iter = list(splitter.split(X_lab, y_multi))

    # weights（balanced）
    sw_multi = make_sample_weights(y_str) if USE_SAMPLE_WEIGHTS else np.ones_like(y_multi, dtype=float)
    yb_lbl = np.where(y_bin == 1, POS_LABEL, "nonT")
    sw_bin  = make_sample_weights(yb_lbl) if USE_SAMPLE_WEIGHTS else np.ones_like(y_bin, dtype=float)

    all_model_outputs: Dict[str, pd.DataFrame] = {}

    for model_name in MODEL_CHOICES:
        rows = []
        fold_probas_multi = []  # (n_unlab, K) per fold
        fold_p_pos        = []  # (n_unlab,) per fold

        for fold, (tr_idx, va_idx) in enumerate(fold_iter, 1):
            X_tr, X_va = X_lab[tr_idx], X_lab[va_idx]
            y_tr_m, y_va_m = y_multi[tr_idx], y_multi[va_idx]
            y_tr_b, y_va_b = y_bin[tr_idx],   y_bin[va_idx]

            # --- 多類 head ---
            clf_multi = build_model(model_name, for_binary=False)
            _fit_with_optional_weight(clf_multi, X_tr, y_tr_m, sw_multi[tr_idx] if USE_SAMPLE_WEIGHTS else None)

            # --- 二元 head ---
            clf_bin = build_model(model_name, for_binary=True)
            _fit_with_optional_weight(clf_bin, X_tr, y_tr_b, sw_bin[tr_idx] if USE_SAMPLE_WEIGHTS else None)

            # ---- Validation metrics (unmask & mask with fixed threshold) ----
            # unmask: 直接 argmax
            proba_va = _predict_proba(clf_multi, X_va)        # (n_va, K)
            yhat_unmask = np.argmax(proba_va, axis=1)

            # mask: 用二元 p_pos >= 0.55 覆寫為 POS_LABEL
            ppos_va = _predict_proba(clf_bin, X_va)           # (n_va, 2)
            ppos_va = ppos_va[:, 1] if ppos_va.ndim == 2 else ppos_va
            yhat_mask = yhat_unmask.copy()
            yhat_mask[ppos_va >= THRESH_FIXED] = idx_T

            def pack(y_true, y_pred):
                m = metrics_all(y_true, y_pred); m["Score"] = 0.5*(m["ARI"]+m["V"]); return m

            m_base = pack(y_va_m, yhat_unmask)
            m_mask = pack(y_va_m, yhat_mask)

            rows.append({
                "model": model_name,
                "fold": fold,
                "val_n": int(len(va_idx)),
                "UNMASK_ACC": m_base["ACC"],
                "UNMASK_BalACC": m_base["Balanced_ACC"],
                "UNMASK_F1m": m_base["F1_macro"],
                "UNMASK_ARI": m_base["ARI"],
                "UNMASK_V":   m_base["V"],
                "UNMASK_Score": m_base["Score"],
                "MASK_ACC": m_mask["ACC"],
                "MASK_BalACC": m_mask["Balanced_ACC"],
                "MASK_F1m": m_mask["F1_macro"],
                "MASK_ARI": m_mask["ARI"],
                "MASK_V":   m_mask["V"],
                "MASK_Score": m_mask["Score"],
                "ΔScore": m_mask["Score"] - m_base["Score"],
            })

            # ---- Predict unlabeled (per fold) ----
            if len(idx_unlab) > 0:
                proba_unlab = _predict_proba(clf_multi, X_unlab)   # (n_unlab, K)
                fold_probas_multi.append(proba_unlab)

                p_pos_unlab = _predict_proba(clf_bin, X_unlab)     # (n_unlab, 2)
                p_pos_unlab = p_pos_unlab[:, 1] if p_pos_unlab.ndim == 2 else p_pos_unlab
                fold_p_pos.append(p_pos_unlab)

        # ---- CV summary for the model ----
        df_cv = pd.DataFrame(rows)
        mean_row = {c:(df_cv[c].mean() if c not in ["model","fold"] else (model_name if c=="model" else "mean")) for c in df_cv.columns}
        df_cv = pd.concat([df_cv, pd.DataFrame([mean_row])], ignore_index=True)
        print(f"\n[CV metrics — {model_name}]")
        print(df_cv.round(4).to_string(index=False))

        # ---- Fold-mean predictions for UNLABELED ----
        if len(idx_unlab) > 0 and len(fold_probas_multi) > 0:
            proba_mean = np.mean(np.stack(fold_probas_multi, axis=0), axis=0)  # (n_test, K)
            ppos_mean  = np.mean(np.stack(fold_p_pos, axis=0), axis=0)         # (n_test,)

            yhat_unmask = np.argmax(proba_mean, axis=1)
            yhat_mask   = yhat_unmask.copy()
            yhat_mask[ppos_mean >= THRESH_FIXED] = idx_T

            out = pd.DataFrame({"cell_id": idx_unlab})
            out["pred_unmask"] = le.inverse_transform(yhat_unmask)
            out["pred_final"]  = le.inverse_transform(yhat_mask)
            out["p_pos_mean"]  = ppos_mean
            out["thr_used"]    = THRESH_FIXED
            for i, cls in enumerate(classes):
                out[f"proba_{cls}"] = proba_mean[:, i]

            print(f"\n[Unlabeled prediction dist — {model_name} | final(masked)]")
            print(out["pred_final"].value_counts(normalize=True).sort_index().mul(100).round(2).astype(str) + "%")
            all_model_outputs[model_name] = out

    return all_model_outputs

# ---------------- Main ----------------
if __name__ == "__main__":
    for name, fname in FILES.items():
        print(f"\n=== Multiclass+Binary Model Zoo — {name} ===")
        df = load_full(os.path.join(IN_DIR, fname))
        outputs = run_kfold_and_predict(df, fname)



=== Multiclass+Binary Model Zoo — Combined (z-score) ===


C:\Users\wani\AppData\Local\Temp\ipykernel_32952\140871484.py:52: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, index_col=0)
c:\Users\wani\AppData\Local\anaconda3\envs\ML4G_project2\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\Users\wani\AppData\Local\anaconda3\envs\ML4G_project2\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\Users\wani\AppData\Local\anaconda3\envs\ML4G_project2\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecat


[CV metrics — logreg_softmax]
         model fold  val_n  UNMASK_ACC  UNMASK_BalACC  UNMASK_F1m  UNMASK_ARI  UNMASK_V  UNMASK_Score  MASK_ACC  MASK_BalACC  MASK_F1m  MASK_ARI  MASK_V  MASK_Score  ΔScore
logreg_softmax    1 6475.0      0.8947         0.9089      0.8498      0.7737    0.7880        0.7808    0.9283       0.8494    0.8574    0.8543  0.8243      0.8393  0.0585
logreg_softmax    2 6475.0      0.8882         0.9084      0.8500      0.7542    0.7806        0.7674    0.9257       0.8494    0.8572    0.8446  0.8178      0.8312  0.0639
logreg_softmax    3 6475.0      0.8975         0.9229      0.8597      0.7740    0.7923        0.7832    0.9294       0.8583    0.8671    0.8523  0.8260      0.8392  0.0560
logreg_softmax    4 6475.0      0.9013         0.9190      0.8610      0.7852    0.7942        0.7897    0.9341       0.8559    0.8675    0.8629  0.8316      0.8473  0.0576
logreg_softmax    5 6474.0      0.8982         0.9227      0.8652      0.7718    0.7984        0.7851   

In [17]:
outputs

{'logreg_softmax':                           cell_id    pred_unmask     pred_final  p_pos_mean  \
 0       AAACCCAAGGAGGCAG-1_5-test              T              T    0.984267   
 1       AAACCCAAGTTGCGCC-1_5-test              T              T    0.996792   
 2       AAACCCACACGGATCC-1_5-test              T              T    0.991180   
 3       AAACCCACATCGGAAG-1_5-test  Myofibroblast              T    0.973026   
 4       AAACCCAGTGCGAGTA-1_5-test              T              T    0.991257   
 ...                           ...            ...            ...         ...   
 18611  TTTGGTTCATTGAAGA-1_10-test              T              T    0.899578   
 18612  TTTGGTTGTTGTCCCT-1_10-test     Fibroblast     Fibroblast    0.000233   
 18613  TTTGGTTGTTTGACAC-1_10-test             NK             NK    0.052425   
 18614  TTTGTTGAGGGTCAAC-1_10-test              T              T    0.986488   
 18615  TTTGTTGCATGGAGAC-1_10-test  Myofibroblast  Myofibroblast    0.000016   
 
        thr_used    

In [18]:
# file: scripts/lgbm_two_stage_mask_vs_unmask_with_ridge_ensemble.py
# -*- coding: utf-8 -*-
"""
Two-Stage CV: LGBM vs Ridge(alpha=0.01) vs Mean-ensemble
- Stage-1 Binary: LGBM(binary), Ridge(alpha=0.01) -> p_pos
- Stage-2 Multiclass: LGBM(multiclass), Ridge(alpha=0.01) -> per-class proba
- For each variant (LGBM / Ridge / Mean):
    * Pick threshold on validation by grid (0.1..0.9) using its binary p_pos
    * base: argmax(multiclass proba)
    * mask: override to POS_LABEL when p_pos >= thr*
- Report metrics for base/mask per fold; add mean row per variant
"""
import os
import numpy as np
import pandas as pd
from typing import Tuple, Dict, List, Optional
from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    adjusted_rand_score, v_measure_score
)
import lightgbm as lgb
from sklearn.linear_model import RidgeClassifier

# ---------------- Config ----------------
IN_DIR = "../outputs_features_csv"
FILES = {
    "Combined (z-score)": "features_combined_z-score.csv",
}
LABEL_COL = "cell_type"
POS_LABEL = "T"
OPTIONAL_GROUP_COLS = ["Patient", "Sample", "set"]
N_SPLITS  = 5
SEED      = 42

# LightGBM
EARLY_STOP_ROUNDS = 50
LGB_PARAMS_MULTI = dict(
    objective="multiclass",
    learning_rate=0.05,
    num_leaves=63,
    min_data_in_leaf=20,
    n_estimators=5000,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=SEED,
    n_jobs=-1,
)
LGB_PARAMS_BIN = dict(
    objective="binary",
    learning_rate=0.05,
    num_leaves=63,
    min_data_in_leaf=20,
    n_estimators=5000,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=SEED,
    n_jobs=-1,
)

# Ridge（固定 alpha=0.01）
RIDGE_ALPHA = 1

# Imbalance controls
USE_SAMPLE_WEIGHTS = True
WEIGHT_SMOOTH_ALPHA = 0.0
WEIGHT_MAX = 5.0

# Threshold grid
THRESH_GRID = np.arange(0.1, 1.0, 0.1)

# ---------------- Utils ----------------
def _sigmoid(x: np.ndarray) -> np.ndarray:
    x = np.clip(x, -50, 50)  # 數值穩定
    return 1.0 / (1.0 + np.exp(-x))

def _softmax(logits: np.ndarray, axis: int = 1) -> np.ndarray:
    z = logits - np.max(logits, axis=axis, keepdims=True)
    z = np.clip(z, -50, 50)
    exp = np.exp(z)
    return exp / np.sum(exp, axis=axis, keepdims=True)

def load_features(path: str, label_col: str = LABEL_COL,
                  optional_group_cols: List[str] = OPTIONAL_GROUP_COLS
                 ) -> Tuple[np.ndarray, np.ndarray, Optional[np.ndarray], List[str]]:
    df = pd.read_csv(path, index_col=0)
    if label_col not in df.columns:
        raise ValueError(f"{os.path.basename(path)} 缺少標籤欄位 '{label_col}'")
    lbl = df[label_col]
    valid_mask = lbl.notna() & lbl.astype(str).str.strip().ne("") & lbl.astype(str).str.lower().ne("nan")
    if valid_mask.sum() == 0:
        raise ValueError(f"{os.path.basename(path)} 內沒有任何帶標籤的資料列。")
    if valid_mask.sum() < len(df):
        print(f"[Info] {os.path.basename(path)} 移除無標籤列：{len(df) - int(valid_mask.sum())} / {len(df)}")
    df = df.loc[valid_mask].copy()

    group = None
    for c in optional_group_cols:
        if c in df.columns:
            group = df[c].astype(str).values
            break

    y_str = df[label_col].astype(str).values
    X_df = df.select_dtypes(include=[np.number]).copy()
    nunique = X_df.nunique(dropna=True)
    keep = (nunique > 1)
    if keep.sum() == 0:
        raise ValueError(f"{os.path.basename(path)} 沒有可用的數值特徵。")
    X_df = X_df.loc[:, keep]

    X = X_df.to_numpy(dtype=float)
    feat_names = X_df.columns.tolist()
    return X, y_str, group, feat_names

def make_sample_weights(y_str: np.ndarray,
                        smooth_alpha: float = WEIGHT_SMOOTH_ALPHA,
                        max_w: Optional[float] = WEIGHT_MAX) -> np.ndarray:
    vc = pd.Series(y_str).value_counts()
    K = float(vc.shape[0]); N = float(len(y_str))
    denom = vc + smooth_alpha
    w_c = (N / (K * denom)).to_dict()
    w = np.array([w_c[y] for y in y_str], dtype=float)
    if max_w is not None:
        w = np.minimum(w, float(max_w))
    return w

def metrics_all(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    return dict(
        ACC=accuracy_score(y_true, y_pred),
        Balanced_ACC=balanced_accuracy_score(y_true, y_pred),
        F1_macro=f1_score(y_true, y_pred, average="macro"),
        ARI=adjusted_rand_score(y_true, y_pred),
        V=v_measure_score(y_true, y_pred),
    )

def pick_best_threshold(p_pos: np.ndarray, y_true_bin: np.ndarray, grid: np.ndarray) -> Tuple[float, float]:
    best_thr, best_acc = 0.5, -1.0
    for t in grid:
        pred = (p_pos >= t).astype(int)
        acc = accuracy_score(y_true_bin, pred)
        if acc > best_acc:
            best_acc, best_thr = acc, t
    return float(best_thr), float(best_acc)

# ---------------- Core ----------------
def run_two_stage_cv_with_ridge_ensemble(X: np.ndarray, y_str: np.ndarray, groups: Optional[np.ndarray]) -> pd.DataFrame:
    le = LabelEncoder().fit(y_str)
    y_multi = le.transform(y_str)
    y_bin = (y_str == POS_LABEL).astype(int)
    classes = le.classes_
    if POS_LABEL not in classes:
        raise ValueError(f"POS_LABEL '{POS_LABEL}' 不存在於標籤：{classes.tolist()}")
    idx_T = int(np.where(classes == POS_LABEL)[0][0])
    K = len(classes)

    if groups is not None:
        splitter = GroupKFold(n_splits=N_SPLITS)
        split_iter = splitter.split(X, y_multi, groups)
    else:
        splitter = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
        split_iter = splitter.split(X, y_multi)

    sw_multi = make_sample_weights(y_str) if USE_SAMPLE_WEIGHTS else np.ones_like(y_multi, dtype=float)
    sw_bin   = make_sample_weights(np.where(y_bin==1, POS_LABEL, "nonT")) if USE_SAMPLE_WEIGHTS else np.ones_like(y_bin, dtype=float)

    rows = []

    for fold, (tr_idx, va_idx) in enumerate(split_iter, 1):
        X_tr, X_va = X[tr_idx], X[va_idx]
        y_tr_multi, y_va_multi = y_multi[tr_idx], y_multi[va_idx]
        y_tr_bin,   y_va_bin   = y_bin[tr_idx],   y_bin[va_idx]
        sw_tr_multi = sw_multi[tr_idx] if USE_SAMPLE_WEIGHTS else None
        sw_tr_bin   = sw_bin[tr_idx]   if USE_SAMPLE_WEIGHTS else None

        # ===== LGBM: Binary =====
        bin_lgb = lgb.LGBMClassifier(**{**LGB_PARAMS_BIN})
        bin_lgb.fit(
            X_tr, y_tr_bin,
            sample_weight=sw_tr_bin,
            eval_set=[(X_va, y_va_bin)],
            eval_metric="logloss",
            callbacks=[lgb.early_stopping(stopping_rounds=EARLY_STOP_ROUNDS, verbose=False)],
        )
        best_it_bin = bin_lgb.best_iteration_ or LGB_PARAMS_BIN["n_estimators"]
        p_pos_lgb = bin_lgb.predict_proba(X_va, num_iteration=best_it_bin)[:, 1]

        # ===== LGBM: Multiclass =====
        multi_lgb = lgb.LGBMClassifier(num_class=K, **LGB_PARAMS_MULTI)
        multi_lgb.fit(
            X_tr, y_tr_multi,
            sample_weight=sw_tr_multi,
            eval_set=[(X_va, y_va_multi)],
            eval_metric="multi_logloss",
            callbacks=[lgb.early_stopping(stopping_rounds=EARLY_STOP_ROUNDS, verbose=False)],
        )
        best_it_multi = multi_lgb.best_iteration_ or LGB_PARAMS_MULTI["n_estimators"]
        proba_lgb = multi_lgb.predict_proba(X_va, num_iteration=best_it_multi)  # (n_va, K)

        # ===== Ridge(alpha=0.01): Binary =====
        bin_ridge = RidgeClassifier(alpha=RIDGE_ALPHA)
        try:
            bin_ridge.fit(X_tr, y_tr_bin, sample_weight=sw_tr_bin)
        except TypeError:
            bin_ridge.fit(X_tr, y_tr_bin)
        p_pos_ridge = _sigmoid(bin_ridge.decision_function(X_va).astype(float))

        # ===== Ridge(alpha=0.01): Multiclass =====
        multi_ridge = RidgeClassifier(alpha=RIDGE_ALPHA)
        try:
            multi_ridge.fit(X_tr, y_tr_multi, sample_weight=sw_tr_multi)
        except TypeError:
            multi_ridge.fit(X_tr, y_tr_multi)
        logits_va = multi_ridge.decision_function(X_va)
        logits_va = np.asarray(logits_va)
        if logits_va.ndim == 1:
            logits_va = logits_va.reshape(-1, 1)
        proba_ridge = _softmax(logits_va, axis=1)

        # ===== 3 variants: LGBM / Ridge / Mean =====
        variants = {
            "lgbm": {
                "p_pos": p_pos_lgb,
                "proba": proba_lgb,
                "best_iter_bin": int(best_it_bin),
                "best_iter_multi": int(best_it_multi),
            },
            "ridge(alpha=0.01)": {
                "p_pos": p_pos_ridge,
                "proba": proba_ridge,
                "best_iter_bin": 0,
                "best_iter_multi": 0,
            },
            "mean(lgbm+ridge)": {
                "p_pos": (p_pos_lgb + p_pos_ridge) / 2.0,
                "proba": (proba_lgb + proba_ridge) / 2.0,
                "best_iter_bin": int(best_it_bin),
                "best_iter_multi": int(best_it_multi),
            },
        }

        for name, pack in variants.items():
            p_pos = pack["p_pos"]
            proba = pack["proba"]

            # threshold per-variant
            thr_star, acc_bin = pick_best_threshold(p_pos, y_va_bin, THRESH_GRID)

            # base
            yhat_base = np.argmax(proba, axis=1)

            # mask
            yhat_mask = yhat_base.copy()
            yhat_mask[p_pos >= thr_star] = idx_T

            m_base = metrics_all(y_va_multi, yhat_base)
            m_mask = metrics_all(y_va_multi, yhat_mask)

            rows.append({
                "model_variant": name,
                "fold": fold,
                "val_n": int(len(va_idx)),
                "thr_star": float(thr_star),
                "bin_ACC_at_thr": float(acc_bin),
                "base_ACC": m_base["ACC"],
                "base_BalACC": m_base["Balanced_ACC"],
                "base_F1m": m_base["F1_macro"],
                "base_ARI": m_base["ARI"],
                "base_V":   m_base["V"],
                "base_Score": 0.5*(m_base["ARI"] + m_base["V"]),
                "mask_ACC": m_mask["ACC"],
                "mask_BalACC": m_mask["Balanced_ACC"],
                "mask_F1m": m_mask["F1_macro"],
                "mask_ARI": m_mask["ARI"],
                "mask_V":   m_mask["V"],
                "mask_Score": 0.5*(m_mask["ARI"] + m_mask["V"]),
                "delta_Score": 0.5*(m_mask["ARI"] + m_mask["V"]) - 0.5*(m_base["ARI"] + m_base["V"]),
                "best_iter_bin": pack["best_iter_bin"],
                "best_iter_multi": pack["best_iter_multi"],
            })

    # 彙總：為每個 model_variant 增加一列 mean
    df = pd.DataFrame(rows)
    mean_rows = []
    for name, sub in df.groupby("model_variant"):
        mean_row = {
            "model_variant": name,
            "fold": "mean",
            "val_n": int(np.mean(sub["val_n"])),
            "thr_star": float(np.mean(sub["thr_star"])),
            "bin_ACC_at_thr": float(np.mean(sub["bin_ACC_at_thr"])),
            "base_ACC": float(np.mean(sub["base_ACC"])),
            "base_BalACC": float(np.mean(sub["base_BalACC"])),
            "base_F1m": float(np.mean(sub["base_F1m"])),
            "base_ARI": float(np.mean(sub["base_ARI"])),
            "base_V":   float(np.mean(sub["base_V"])),
            "base_Score": float(np.mean(sub["base_Score"])),
            "mask_ACC": float(np.mean(sub["mask_ACC"])),
            "mask_BalACC": float(np.mean(sub["mask_BalACC"])),
            "mask_F1m": float(np.mean(sub["mask_F1m"])),
            "mask_ARI": float(np.mean(sub["mask_ARI"])),
            "mask_V":   float(np.mean(sub["mask_V"])),
            "mask_Score": float(np.mean(sub["mask_Score"])),
            "delta_Score": float(np.mean(sub["delta_Score"])),
            "best_iter_bin": int(np.round(np.mean(sub["best_iter_bin"]))),
            "best_iter_multi": int(np.round(np.mean(sub["best_iter_multi"]))),
        }
        mean_rows.append(mean_row)
    df = pd.concat([df, pd.DataFrame(mean_rows)], ignore_index=True)
    return df

# ---------------- Run ----------------
available: Dict[str, str] = {}
for name, fname in FILES.items():
    path = os.path.join(IN_DIR, fname)
    if os.path.exists(path):
        available[name] = path

if not available:
    raise FileNotFoundError(f"找不到任何特徵檔於 {IN_DIR}：{list(FILES.values())}")

for name, path in available.items():
    print(f"\n=== Two-Stage CV (LGBM vs Ridge vs Mean) on: {name} ===")
    X, y_str, groups, feat_names = load_features(path)
    print(f"Loaded: {os.path.basename(path)} | X={X.shape} | classes={len(np.unique(y_str))} | features={len(feat_names)}"
          + (f" | groups={groups.dtype.name}" if groups is not None else " | groups=None"))

    df = run_two_stage_cv_with_ridge_ensemble(X, y_str, groups)
    # 先列出各 variant 的 mean
    print("\n[Mean per model_variant]")
    print(df[df["fold"]=="mean"].round(4).to_string(index=False))
    # 若需要完整每折細節：
    print("\n[All folds]")
    print(df[df["fold"]!="mean"].round(4).to_string(index=False))



=== Two-Stage CV (LGBM vs Ridge vs Mean) on: Combined (z-score) ===


C:\Users\wani\AppData\Local\Temp\ipykernel_32952\2256028751.py:87: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, index_col=0)


[Info] features_combined_z-score.csv 移除無標籤列：18616 / 50990
Loaded: features_combined_z-score.csv | X=(32374, 101) | classes=9 | features=101 | groups=None
[LightGBM] [Warning] min_data_in_leaf is set=20, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=20
[LightGBM] [Warning] min_data_in_leaf is set=20, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=20
[LightGBM] [Info] Number of positive: 13340, number of negative: 12559
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.011589 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 25755
[LightGBM] [Info] Number of data points in the train set: 25899, number of used features: 101
[LightGBM] [Warning] min_data_in_leaf is set=20, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=20
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.499973 -> initscore=-0.000108
[LightGBM] [Info] Start training from 

c:\Users\wani\AppData\Local\anaconda3\envs\ML4G_project2\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] min_data_in_leaf is set=20, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=20
[LightGBM] [Warning] min_data_in_leaf is set=20, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=20
[LightGBM] [Warning] min_data_in_leaf is set=20, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=20
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.007423 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 25755
[LightGBM] [Info] Number of data points in the train set: 25899, number of used features: 101
[LightGBM] [Warning] min_data_in_leaf is set=20, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=20
[LightGBM] [Info] Start training from score -2.178453
[LightGBM] [Info] Start training from score -2.362116
[LightGBM] [Info] Start training from score -2.178800
[LightGBM] [Info] Start training from score -2.178062
[Ligh

...

c:\Users\wani\AppData\Local\anaconda3\envs\ML4G_project2\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] min_data_in_leaf is set=20, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=20
[LightGBM] [Warning] min_data_in_leaf is set=20, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=20
[LightGBM] [Warning] min_data_in_leaf is set=20, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=20
[LightGBM] [Info] Number of positive: 13341, number of negative: 12559
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.007537 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 25755
[LightGBM] [Info] Number of data points in the train set: 25900, number of used features: 101
[LightGBM] [Warning] min_data_in_leaf is set=20, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=20
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.499992 -> initscore=-0.000033
[LightGBM] [Info] Start training from score -0.000033


c:\Users\wani\AppData\Local\anaconda3\envs\ML4G_project2\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] min_data_in_leaf is set=20, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=20
[LightGBM] [Warning] min_data_in_leaf is set=20, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=20
[LightGBM] [Warning] min_data_in_leaf is set=20, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=20
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.007711 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 25755
[LightGBM] [Info] Number of data points in the train set: 25900, number of used features: 101
[LightGBM] [Warning] min_data_in_leaf is set=20, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=20
[LightGBM] [Info] Start training from score -2.178479
[LightGBM] [Info] Start training from score -2.360056
[LightGBM] [Info] Start training from score -2.178826
[LightGBM] [Info] Start training from score -2.178088
[Ligh

c:\Users\wani\AppData\Local\anaconda3\envs\ML4G_project2\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [10]:
# file: scripts/ridge_two_stage_kfold_fixedthr_predict_test.py
# -*- coding: utf-8 -*-
"""
Two-Stage KFold (Fixed Threshold) using RidgeClassifier instead of LightGBM.
- Binary stage: RidgeClassifier (POS_LABEL vs others), decision_function -> sigmoid -> pT
- Multiclass stage: RidgeClassifier (OvR), decision_function -> softmax -> per-class scores
- Keep original CV metrics, sample-weighting, grouping, outputs
"""
import os
import numpy as np
import pandas as pd
from typing import Tuple, Dict, List, Optional
from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    adjusted_rand_score, v_measure_score
)
from sklearn.linear_model import RidgeClassifier
from sklearn.pipeline import make_pipeline

# ---------------- Config ----------------
IN_DIR = "../outputs_features_csv"
FILES = {
    "Combined (z-score)": "features_combined_z-score.csv",
}
LABEL_COL = "cell_type"
POS_LABEL = "T"
OPTIONAL_GROUP_COLS = ["Patient", "Sample", "set"]
N_SPLITS = 5
SEED = 42

# 固定門檻
FIXED_THRESHOLD = 0.5

# Ridge 參數（可依需要調整 alpha）
RIDGE_PARAMS_MULTI = dict(alpha=1.0, random_state=None)
RIDGE_PARAMS_BIN   = dict(alpha=1.0, random_state=None)

# Imbalance：樣本權重（不改分布）
USE_SAMPLE_WEIGHTS = True
WEIGHT_SMOOTH_ALPHA = 0.0
WEIGHT_MAX = 5.0

# 是否加入標準化（若輸入已是 z-score 可關閉）
USE_STANDARD_SCALER = False  # 保留與原檔一致性；若特徵未標準化建議 True

# ---------------- Utils ----------------
def load_full(path: str) -> pd.DataFrame:
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    df = pd.read_csv(path, index_col=0)
    if df.empty:
        raise ValueError(f"Empty: {path}")
    return df

def split_masks(df: pd.DataFrame, label_col: str) -> Tuple[pd.Series, pd.Series]:
    if label_col not in df.columns:
        return pd.Series(False, index=df.index), pd.Series(True, index=df.index)
    lbl = df[label_col]
    is_lab = (lbl.notna()
              & lbl.astype(str).str.strip().ne("")
              & lbl.astype(str).str.lower().ne("nan"))
    return is_lab, ~is_lab

def numeric_feature_matrix(df: pd.DataFrame) -> pd.DataFrame:
    Xdf = df.select_dtypes(include=[np.number]).copy()
    nunique = Xdf.nunique(dropna=True)
    keep = (nunique > 1)
    Xdf = Xdf.loc[:, keep]
    if Xdf.shape[1] == 0:
        raise ValueError("No usable numeric features (all constant/NaN).")
    return Xdf.fillna(0.0)

def groups_from(df: pd.DataFrame, cols: List[str]) -> Optional[np.ndarray]:
    for c in cols:
        if c in df.columns:
            return df[c].astype(str).values
    return None

def make_sample_weights(y_like: np.ndarray,
                        smooth_alpha: float = WEIGHT_SMOOTH_ALPHA,
                        max_w: Optional[float] = WEIGHT_MAX) -> np.ndarray:
    vc = pd.Series(y_like).value_counts()
    K = float(vc.shape[0]); N = float(len(y_like))
    denom = vc + smooth_alpha
    w_c = (N / (K * denom)).to_dict()
    w = np.array([w_c[y] for y in y_like], dtype=float)
    if max_w is not None:
        w = np.minimum(w, float(max_w))
    return w

def ensure_prob_matrix(proba, K: int) -> np.ndarray:
    arr = np.asarray(proba, dtype=float)
    if arr.ndim == 1:
        arr = arr.reshape(-1, 1)
    if arr.shape[1] != K:
        pass
    return arr

def metrics_all(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    return dict(
        ACC=accuracy_score(y_true, y_pred),
        Balanced_ACC=balanced_accuracy_score(y_true, y_pred),
        F1_macro=f1_score(y_true, y_pred, average="macro"),
        ARI=adjusted_rand_score(y_true, y_pred),
        V=v_measure_score(y_true, y_pred),
    )

# --- score->prob helpers（Ridge 無 predict_proba，用這兩個轉換） ---
def _sigmoid(x: np.ndarray) -> np.ndarray:
    x = np.clip(x, -50, 50)  # 穩定
    return 1.0 / (1.0 + np.exp(-x))

def _softmax(logits: np.ndarray, axis: int = 1) -> np.ndarray:
    z = logits - np.max(logits, axis=axis, keepdims=True)
    exp = np.exp(np.clip(z, -50, 50))
    return exp / np.sum(exp, axis=axis, keepdims=True)

def _make_ridge_pipeline(params: Dict) -> any:
    clf = RidgeClassifier(**{k:v for k,v in params.items() if k != "random_state"})
    if USE_STANDARD_SCALER:
        return make_pipeline(StandardScaler(with_mean=False), clf)
    return clf

# ---------------- Per-file run ----------------
def run_kfold_and_predict(df: pd.DataFrame, fname: str) -> pd.DataFrame:
    is_lab, is_unlab = split_masks(df, LABEL_COL)
    if not is_lab.any():
        raise ValueError(f"{fname}: no labeled rows for training.")

    X_all_df = numeric_feature_matrix(df)
    X_all = X_all_df.to_numpy(dtype=float)
    y_str_all = df[LABEL_COL].astype(str).values
    idx_all = df.index.to_numpy()

    # labeled / unlabeled
    X_lab = X_all[is_lab.values]
    y_str = y_str_all[is_lab.values]
    idx_lab = idx_all[is_lab.values]

    X_unlab = X_all[is_unlab.values]
    idx_unlab = idx_all[is_unlab.values]

    # encoders & constants
    le = LabelEncoder().fit(y_str)
    classes = le.classes_
    if POS_LABEL not in classes:
        raise ValueError(f"{fname}: POS_LABEL '{POS_LABEL}' not in labels {classes.tolist()}")
    idx_T = int(np.where(classes == POS_LABEL)[0][0])
    y = le.transform(y_str)
    y_bin = (y_str == POS_LABEL).astype(int)
    K = len(classes)

    # groups (only for labeled)
    groups = groups_from(df.loc[idx_lab], OPTIONAL_GROUP_COLS)

    # split
    if groups is not None:
        splitter = GroupKFold(n_splits=N_SPLITS)
        split_iter = splitter.split(X_lab, y, groups)
    else:
        splitter = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
        split_iter = splitter.split(X_lab, y)

    # weights
    sw_multi = make_sample_weights(y_str) if USE_SAMPLE_WEIGHTS else np.ones_like(y, dtype=float)
    yb_lbl = np.where(y_bin == 1, POS_LABEL, "nonT")
    sw_bin  = make_sample_weights(yb_lbl) if USE_SAMPLE_WEIGHTS else np.ones_like(y_bin, dtype=float)

    # storage for fold-wise test predictions
    test_bin_probs: List[np.ndarray] = []   # each (n_test,)
    test_multi_probs: List[np.ndarray] = [] # each (n_test, K)

    rows = []

    for fold, (tr_idx, va_idx) in enumerate(split_iter, 1):
        X_tr, X_va = X_lab[tr_idx], X_lab[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]
        yb_tr, yb_va = y_bin[tr_idx], y_bin[va_idx]

        # ===== Binary (Ridge) =====
        bin_clf = _make_ridge_pipeline(RIDGE_PARAMS_BIN)
        # 為了 pipeline 的 sample_weight：取最後一步
        if isinstance(bin_clf, tuple) or hasattr(bin_clf, "steps"):
            # sklearn Pipeline
            bin_clf.fit(X_tr, yb_tr, ridgeclassifier__sample_weight=(sw_bin[tr_idx] if USE_SAMPLE_WEIGHTS else None))
        else:
            bin_clf.fit(X_tr, yb_tr, sample_weight=(sw_bin[tr_idx] if USE_SAMPLE_WEIGHTS else None))

        # decision_function -> sigmoid -> p_va
        if hasattr(bin_clf, "decision_function"):
            df_va_scores = bin_clf.decision_function(X_va)
        else:
            # Pipeline
            df_va_scores = bin_clf[-1].decision_function(bin_clf[:-1].transform(X_va)) if hasattr(bin_clf, "__getitem__") else bin_clf.decision_function(X_va)
        p_va = _sigmoid(np.asarray(df_va_scores).ravel())
        mask_va = (p_va >= FIXED_THRESHOLD)

        # ===== Multiclass (Ridge) =====
        multi_clf = _make_ridge_pipeline(RIDGE_PARAMS_MULTI)
        if isinstance(multi_clf, tuple) or hasattr(multi_clf, "steps"):
            multi_clf.fit(X_tr, y_tr, ridgeclassifier__sample_weight=(sw_multi[tr_idx] if USE_SAMPLE_WEIGHTS else None))
        else:
            multi_clf.fit(X_tr, y_tr, sample_weight=(sw_multi[tr_idx] if USE_SAMPLE_WEIGHTS else None))

        # decision_function -> softmax -> proba_va
        if hasattr(multi_clf, "decision_function"):
            logits_va = multi_clf.decision_function(X_va)
        else:
            logits_va = multi_clf[-1].decision_function(multi_clf[:-1].transform(X_va)) if hasattr(multi_clf, "__getitem__") else multi_clf.decision_function(X_va)
        logits_va = np.asarray(logits_va)
        if logits_va.ndim == 1:
            logits_va = logits_va.reshape(-1, 1)
        proba_va = _softmax(logits_va, axis=1)

        yhat_unmask = np.argmax(proba_va, axis=1)
        yhat_mask = yhat_unmask.copy()
        yhat_mask[mask_va] = idx_T

        def pack(y_true, y_pred):
            m = metrics_all(y_true, y_pred); m["Score"] = 0.5*(m["ARI"]+m["V"]); return m
        m_base = pack(y_va, yhat_unmask)
        m_mask = pack(y_va, yhat_mask)

        rows.append({
            "fold": fold,
            "val_n": int(len(va_idx)),
            "thr_fixed": FIXED_THRESHOLD,
            "UNMASK_ACC": m_base["ACC"],
            "UNMASK_BalACC": m_base["Balanced_ACC"],
            "UNMASK_F1m": m_base["F1_macro"],
            "UNMASK_ARI": m_base["ARI"],
            "UNMASK_V":   m_base["V"],
            "UNMASK_Score": m_base["Score"],
            "MASK_ACC": m_mask["ACC"],
            "MASK_BalACC": m_mask["Balanced_ACC"],
            "MASK_F1m": m_mask["F1_macro"],
            "MASK_ARI": m_mask["ARI"],
            "MASK_V":   m_mask["V"],
            "MASK_Score": m_mask["Score"],
            "ΔScore": m_mask["Score"] - m_base["Score"],
        })

        # ===== Predict ALL test rows in this fold =====
        if len(idx_unlab) > 0:
            # binary
            if hasattr(bin_clf, "decision_function"):
                df_test_scores = bin_clf.decision_function(X_unlab)
            else:
                df_test_scores = bin_clf[-1].decision_function(bin_clf[:-1].transform(X_unlab)) if hasattr(bin_clf, "__getitem__") else bin_clf.decision_function(X_unlab)
            p_test = _sigmoid(np.asarray(df_test_scores).ravel())
            test_bin_probs.append(p_test)

            # multiclass
            if hasattr(multi_clf, "decision_function"):
                logits_test = multi_clf.decision_function(X_unlab)
            else:
                logits_test = multi_clf[-1].decision_function(multi_clf[:-1].transform(X_unlab)) if hasattr(multi_clf, "__getitem__") else multi_clf.decision_function(X_unlab)
            logits_test = np.asarray(logits_test)
            if logits_test.ndim == 1:
                logits_test = logits_test.reshape(-1, 1)
            proba_test = _softmax(logits_test, axis=1)
            test_multi_probs.append(proba_test)

    # ---- CV summary ----
    df_cv = pd.DataFrame(rows)
    mean_row = {c:(df_cv[c].mean() if c not in ["fold"] else "mean") for c in df_cv.columns}
    df_cv = pd.concat([df_cv, pd.DataFrame([mean_row])], ignore_index=True)
    print("\n[CV metrics]")
    print(df_cv.to_string(index=False))

    # ---- Fold-mean predictions for TEST ----
    if len(idx_unlab) > 0 and len(test_bin_probs) > 0 and len(test_multi_probs) > 0:
        pT_mean = np.mean(np.vstack(test_bin_probs), axis=0)                 # (n_test,)
        proba_mean = np.mean(np.stack(test_multi_probs, axis=0), axis=0)     # (n_test, K)

        yhat_unmask = np.argmax(proba_mean, axis=1)
        yhat_mask = yhat_unmask.copy()
        yhat_mask[pT_mean >= FIXED_THRESHOLD] = idx_T

        out = pd.DataFrame({"cell_id": idx_unlab, "pT_mean": pT_mean, "thr_used": FIXED_THRESHOLD})
        out["pred_unmask"] = le.inverse_transform(yhat_unmask)
        out["pred_final"]  = le.inverse_transform(yhat_mask)
        for i, cls in enumerate(classes):
            out[f"proba_{cls}"] = proba_mean[:, i]

        print(out["pred_final"].value_counts(normalize=True).sort_index().mul(100).round(2).astype(str) + "%")
    else:
        print("\n[Info] No unlabeled rows to predict.")
        out = pd.DataFrame(columns=["cell_id","pT_mean","thr_used","pred_unmask","pred_final"])

    return out

# ---------------- Main ----------------
if __name__ == "__main__":
    for name, fname in FILES.items():
        print(f"\n=== Two-Stage KFold w/ fixed threshold (0.55, Ridge) — {name} ===")
        df = load_full(os.path.join(IN_DIR, fname))
        pred = run_kfold_and_predict(df, fname)



=== Two-Stage KFold w/ fixed threshold (0.55, Ridge) — Combined (z-score) ===


C:\Users\wani\AppData\Local\Temp\ipykernel_32952\2034323649.py:52: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, index_col=0)



[CV metrics]
fold  val_n  thr_fixed  UNMASK_ACC  UNMASK_BalACC  UNMASK_F1m  UNMASK_ARI  UNMASK_V  UNMASK_Score  MASK_ACC  MASK_BalACC  MASK_F1m  MASK_ARI   MASK_V  MASK_Score   ΔScore
   1 6475.0        0.5    0.876911       0.898639    0.842109    0.734236  0.789411      0.761823  0.925251     0.817326  0.829941  0.845010 0.823695    0.834352 0.072529
   2 6475.0        0.5    0.877529       0.902053    0.847869    0.729978  0.790027      0.760003  0.926178     0.819937  0.834806  0.841835 0.826073    0.833954 0.073951
   3 6475.0        0.5    0.882008       0.914558    0.857682    0.736438  0.790978      0.763708  0.928340     0.828775  0.845084  0.842612 0.829275    0.835944 0.072236
   4 6475.0        0.5    0.882934       0.907718    0.854624    0.741480  0.791304      0.766392  0.929266     0.822150  0.834232  0.846874 0.831369    0.839121 0.072729
   5 6474.0        0.5    0.882607       0.914293    0.857036    0.734792  0.800745      0.767768  0.932345     0.831888  0.843810 

In [ ]:
# Jupyter cell: build cluster_membership.csv from in-memory `pred` only
import os
import re
import pandas as pd
import scanpy as sc

# ---- Settings ----
TEST_AD_PATH     = "../test_data/test_adata.h5ad"
OUT_CLUSTER_CSV  = "../workflow/outputs/cluster_membership.csv"
STRIP_TEST_SUFFIX = True  # True: 比對前移除 cell_id 結尾的 "-test"

# ---- Guards ----
if "pred" not in globals() or not isinstance(pred, pd.DataFrame):
    raise RuntimeError("需要記憶體中的 DataFrame `pred`（含 columns: cell_id, pred_final）。")

need_cols = {"cell_id", "pred_final"}
missing = need_cols - set(pred.columns)
if missing:
    raise ValueError(f"`pred` 缺少欄位: {missing}. 需要 {need_cols}")

df_pred = pred[list(need_cols)].copy()

# ---- Normalize cell_id for matching ----
if STRIP_TEST_SUFFIX:
    df_pred["cell_id_std"] = df_pred["cell_id"].astype(str).str.replace(r"-test$", "", regex=True)
else:
    df_pred["cell_id_std"] = df_pred["cell_id"].astype(str)

# ---- Load test obs order ----
ad_te = sc.read_h5ad(TEST_AD_PATH)
test_order = pd.Index(ad_te.obs_names.astype(str))

# ---- Align & reorder to test order ----
pred_idx = pd.Index(df_pred["cell_id_std"])
common   = test_order.intersection(pred_idx)
if common.empty:
    raise ValueError("pred 的 cell_id 與 test obs_names 沒有交集（檢查 -test 尾碼或命名）。")

df_pred_idxed = (
    df_pred.set_index("cell_id_std", drop=False)
           .loc[common, ["cell_id", "pred_final"]]
           .rename_axis("cell_id_std")
           .reset_index(drop=True)
)
# 使輸出的 cell_id 與 test 完全一致（不帶 '-test'）
df_pred_idxed["cell_id"] = common.values

if len(common) < len(test_order):
    missing_n = len(test_order) - len(common)
    print(f"[Info] test n={len(test_order)}，pred 可對齊 n={len(common)}；缺少 {missing_n} 列未在 pred 中。")

# ---- Build cluster mapping (1..K) ----
labels = df_pred_idxed["pred_final"].astype(str)
uniq_labels = sorted(labels.unique().tolist())

use_order = None
if "TARGET_ORDER" in globals():
    to_set = set(TARGET_ORDER)
    if set(uniq_labels).issubset(to_set):
        use_order = [ct for ct in TARGET_ORDER if ct in uniq_labels]
if use_order is None:
    use_order = uniq_labels  # 字母序穩定

label_to_cluster = {lab: i+1 for i, lab in enumerate(use_order)}  # 1..K

# ---- Compose and save cluster_membership ----
df_cluster = pd.DataFrame({
    "index": df_pred_idxed["cell_id"].values,                           # 與 test obs_names 完全一致的順序
    "cluster": labels.map(label_to_cluster).astype(int).values
})
if df_cluster["index"].duplicated().any():
    raise ValueError("cluster_membership 中 index 出現重複，請檢查。")
if (df_cluster["cluster"] <= 0).any():
    raise ValueError("cluster 必須為正整數（>=1）。")

df_cluster.to_csv(OUT_CLUSTER_CSV, index=True)  # 依你的示例，保留 pandas 行索引
print(f"[OK] wrote cluster_membership: {OUT_CLUSTER_CSV} | rows={len(df_cluster)}")

print("\n(label -> cluster) mapping:")
print(pd.Series(label_to_cluster).to_string())

print("\n[cluster_membership head]")
print(df_cluster.head(10).to_csv(index=True))
